In [0]:
%sql
CREATE CATALOG IF NOT EXISTS opensky
MANAGED LOCATION 
'abfss://batch@uralibootcamp.dfs.core.windows.net/managed/opensky';

CREATE SCHEMA IF NOT EXISTS opensky.bronze;

CREATE SCHEMA IF NOT EXISTS  opensky.silver;

CREATE SCHEMA IF NOT EXISTS  opensky.gold;

In [0]:
raw_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/raw/Flights"

checkpoint_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/checkpoint/Flights"

schema_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/schema/Flights"
bronze_path = "abfss://batch@uralibootcamp.dfs.core.windows.net/bronze/lights"

#bronze_table = "opensky.bronze.bronze_flights"

Define OpenSky schema

For Bronze, keep raw data as strings:

In [0]:
from pyspark.sql.types import *

opensky_schema = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("baro_altitude", StringType(), True),
    StructField("velocity", StringType(), True),
    StructField("heading", StringType(), True),
    StructField("on_ground", StringType(), True)
])

Auto Loader batch ingestion

Add ingestion metadata 

For Bronze history tracking:


In [0]:
from pyspark.sql.functions import *

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(raw_path)
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

Write to Bronze Delta table (Batch Mode)

Use availableNow


In [0]:
query = (
    bronze_df.writeStream
    .format("csv")
    .option("header", "true")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .option("path", bronze_path)
    .start()
)

Read all existing CSV files in raw/flights
Load them into Bronze
Stop automatically
Next run only picks new files